# 06 - Outlier Detection & Treatment

## Objective

Detect, analyze, and treat outliers in Marketing Mix Modeling datasets without
removing genuine business events such as festivals or major campaigns.

**Topics**
- IQR Method
- Z-Score
- Modified Z-Score (MAD)
- Isolation Forest
- Winsorization
- Business interpretation


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.stats import zscore
from sklearn.ensemble import IsolationForest

ROOT = Path.cwd()
DATA = ROOT/"data"/"processed"/"marketing_mix_imputed.csv"

df = pd.read_csv(DATA, parse_dates=["Week"])

numeric = df.select_dtypes(include=np.number)

numeric.head()


## 1. Boxplots

In [ ]:

cols=["Sales","Revenue","Google_Search","Meta","TV","Competitor_Spend"]

for c in cols:
    plt.figure(figsize=(8,1.8))
    plt.boxplot(df[c],vert=False)
    plt.title(c)
    plt.show()


## 2. IQR Method

In [ ]:

iqr_summary=[]

for c in numeric.columns:
    q1=df[c].quantile(.25)
    q3=df[c].quantile(.75)
    iqr=q3-q1
    lower=q1-1.5*iqr
    upper=q3+1.5*iqr

    count=((df[c]<lower)|(df[c]>upper)).sum()

    iqr_summary.append({
        "Feature":c,
        "Outliers":count,
        "Lower":round(lower,2),
        "Upper":round(upper,2)
    })

iqr_report=pd.DataFrame(iqr_summary)
display(iqr_report.head(15))


## 3. Z-Score

In [ ]:

z_df=pd.DataFrame(index=df.index)

for c in numeric.columns:
    z_df[c]=np.abs(zscore(df[c]))

z_counts=(z_df>3).sum().sort_values(ascending=False)

display(z_counts.to_frame("ZScore_Outliers"))


## 4. Modified Z-Score (MAD)

In [ ]:

def modified_z(series):
    median=np.median(series)
    mad=np.median(np.abs(series-median))
    if mad==0:
        return np.zeros(len(series))
    return 0.6745*(series-median)/mad

mad_report=[]

for c in numeric.columns:
    score=np.abs(modified_z(df[c]))
    mad_report.append([c,(score>3.5).sum()])

mad_report=pd.DataFrame(mad_report,columns=["Feature","MAD_Outliers"])
display(mad_report)


## 5. Isolation Forest

In [ ]:

iso=IsolationForest(
    contamination=0.03,
    random_state=42
)

pred=iso.fit_predict(numeric)

df["IsolationForest_Outlier"]=pred

print("Detected Outliers:",(pred==-1).sum())


## 6. Winsorization

In [ ]:

winsor=df.copy()

for c in numeric.columns:
    q1=winsor[c].quantile(.25)
    q3=winsor[c].quantile(.75)
    iqr=q3-q1

    lower=q1-1.5*iqr
    upper=q3+1.5*iqr

    winsor[c]=winsor[c].clip(lower,upper)

display(winsor.head())


## 7. Compare Original vs Winsorized

In [ ]:

plt.figure(figsize=(10,4))

plt.hist(df["Sales"],bins=25,alpha=.5,label="Original")
plt.hist(winsor["Sales"],bins=25,alpha=.5,label="Winsorized")

plt.legend()
plt.title("Sales Distribution")
plt.show()


## 8. Business Guidelines

In [ ]:

guidelines=pd.DataFrame({
"Scenario":[
"Black Friday sales spike",
"Diwali campaign",
"System data error",
"Negative media spend",
"Extreme competitor spend"
],
"Action":[
"Keep",
"Keep",
"Remove/Correct",
"Investigate",
"Validate with business team"
]
})

display(guidelines)


In [ ]:

OUTPUT=ROOT/"data"/"processed"/"marketing_mix_outliers_treated.csv"
winsor.to_csv(OUTPUT,index=False)
print("Saved:",OUTPUT)


# Key Takeaways

- Outliers are **not always bad**.
- Large marketing spikes can represent genuine business events.
- Compare multiple detection techniques before deciding on treatment.
- Winsorization is often preferred over deleting records in MMM because it preserves the time series.
